# Titanic — Model Inference

Load the best model from MLflow Model Registry and generate `submission.csv`.

In [38]:
!pip install mlflow category_encoders dagshub 


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [39]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import category_encoders as ce

warnings.filterwarnings('ignore')

## MLflow / DagsHub Setup

Must match the settings used in `model_experiment.ipynb`.

In [40]:
import dagshub

DAGSHUB_USERNAME = "konstantine25b"
DAGSHUB_REPO     = "titanic-for-tutoring"
DAGSHUB_TOKEN    = "2eaae1414498f697a730998443323fc67f26b842"

import os
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

import mlflow
mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow")

print("Tracking URI:", mlflow.get_tracking_uri())


Tracking URI: https://dagshub.com/konstantine25b/titanic-for-tutoring.mlflow


## Load Model from Registry

In [41]:
MODEL_NAME    = "titanic-best-model"
MODEL_VERSION = "latest"   # or a specific version number like "1"

model_uri = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
model     = mlflow.sklearn.load_model(model_uri)
print("Model loaded:", type(model).__name__)

Model loaded: Pipeline


## Load Preprocessing Artifacts

In [42]:
from mlflow.tracking import MlflowClient

client   = MlflowClient()
versions = client.get_latest_versions('titanic-best-model')
run_id   = versions[0].run_id

local_path = mlflow.artifacts.download_artifacts(
    run_id=run_id, artifact_path='preprocessing_artifacts.pkl'
)
with open(local_path, 'rb') as f:
    artifacts = pickle.load(f)

best_encoding = artifacts['best_encoding']
selected_ohe  = artifacts['selected_ohe']
selected_woe  = artifacts['selected_woe']
woe_encoder   = artifacts['woe_encoder']
NUMERIC_COLS  = artifacts['numeric_cols']
CAT_COLS      = artifacts['cat_cols']
WOE_COLS      = artifacts['woe_cols']

print('Artifacts loaded from MLflow run:', run_id)
print('Best encoding:', best_encoding)


Artifacts loaded from MLflow run: 20007cceb5a34532bfa4bf7ceefa0626
Best encoding: OHE


## Preprocessing Pipeline (mirrors model_experiment.ipynb)

In [43]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.drop_duplicates(inplace=True)
    df.fillna(df.median(numeric_only=True), inplace=True)
    return df.reset_index(drop=True)


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['Pclass']   = (df[['Pclass_1','Pclass_2','Pclass_3']]
                      .idxmax(axis=1)
                      .str.replace('Pclass_', '', regex=False)
                      .astype(int))
    df['Title']    = (df[['Title_1','Title_2','Title_3','Title_4']]
                      .idxmax(axis=1)
                      .str.replace('Title_', '', regex=False)
                      .astype(int))
    df['Embarked'] = (df[['Emb_1','Emb_2','Emb_3']]
                      .idxmax(axis=1)
                      .str.replace('Emb_', '', regex=False)
                      .astype(int))
    df['IsAlone']  = (df['Family_size'] == 0).astype(int)
    df.drop(columns=['Pclass_1','Pclass_2','Pclass_3',
                     'Title_1','Title_2','Title_3','Title_4',
                     'Emb_1','Emb_2','Emb_3'], inplace=True)
    return df


def preprocess(df: pd.DataFrame) -> tuple:
    df = clean(df)
    ids = df['PassengerId'].copy() if 'PassengerId' in df.columns else df.index
    df.drop(columns=['PassengerId'], errors='ignore', inplace=True)
    df = add_features(df)

    if best_encoding == 'OHE':
        ohe_dummies = pd.get_dummies(df[CAT_COLS].astype(str), columns=CAT_COLS, drop_first=False)
        X = pd.concat([df[NUMERIC_COLS].reset_index(drop=True), ohe_dummies], axis=1)
        for col in selected_ohe:
            if col not in X.columns:
                X[col] = 0
        X = X[selected_ohe]
    else:
        woe_input = df[NUMERIC_COLS + WOE_COLS].copy().reset_index(drop=True)
        X = woe_encoder.transform(woe_input)[selected_woe]

    return X.values.astype(float), ids.reset_index(drop=True)


## Load Test Data & Generate Submission

In [44]:
test_raw = pd.read_csv('data/test_data.csv', index_col=0)

X_test, passenger_ids = preprocess(test_raw)

print('Test feature matrix shape:', X_test.shape)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': passenger_ids.values,
    'Survived':    predictions
})
submission.to_csv('submission.csv', index=False)
print('submission.csv saved.')
submission.head(10)


Test feature matrix shape: (100, 15)
submission.csv saved.


,PassengerId,Survived
0,792,0
1,793,0
2,794,0
3,795,0
4,796,0
5,797,1
6,798,1
7,799,0
8,800,1
9,801,0


In [45]:
print("Survival rate in submission:", submission['Survived'].mean().round(4))
submission['Survived'].value_counts()

Survival rate in submission: 0.36


Survived
0    64
1    36
Name: count, dtype: int64